# 🏗️ Starter — CNN Mini dari Nol (Bab 5)> Notebook eksperimen. **Prasyarat:** semua 42 test hijau> (`python -m unittest discover -s tests -v`).**Aturan main:** semua keputusan (filter, augmentasi, lr, early stop) diambil di**validation set**. Test set hanya disentuh di Bagian 7, sekali.

## Setup & Peta Data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from nn_mini.data import NILAI_KELAS, muat_semua
from nn_mini.train import train_model

X_train, y_train, X_val, y_val, X_test, y_test = muat_semua()
print('splits:', len(y_train), len(y_val), len(y_test),
      '| vertikal:', int(y_train.sum()), int(y_val.sum()), int(y_test.sum()))


In [ ]:
# Lihat dulu bentuk datanya — gambar 8x8, satu garis terang horizontal/vertikal
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for k, kelas in enumerate([0, 1]):
    contoh = X_train[y_train == kelas][:6]
    for j in range(6):
        axes[k, j].imshow(contoh[j], cmap='gray')
        axes[k, j].set_title(NILAI_KELAS[kelas], fontsize=9)
        axes[k, j].axis('off')
plt.suptitle('Dataset "bars": 0 = horizontal, 1 = vertikal')
plt.tight_layout(); plt.show()


## Bagian 1 — Baseline MLP di Piksel Mentah DuluSebelum CNN, buktikan bahwa MLP yang mengahmali posisi piksel itu rapuh.Pakai `Dense` dari `nn_mini.nnet` (sudah diberikan) — flatten manual + MLP 64-16-1.

In [ ]:
from nn_mini.nnet import Dense, Sequential

Xtr = X_train.reshape(len(X_train), -1)
Xva = X_val.reshape(len(X_val), -1)

mlp = Sequential([Dense(64, activation='relu', seed=1),
                  Dense(16, activation='relu', seed=2),
                  Dense(1, activation='sigmoid', seed=3)])
hist_mlp = train_model(mlp, Xtr, y_train, Xva, y_val,
                       epochs=60, batch_size=32, lr=0.3, seed=0)
print(f'MLP (piksel mentah): val acc = {hist_mlp["val_acc"][-1]:.4f}')
print('-> Catat angka ini. CNN nanti HARUS mengalahkannya.')


**Refleksi 1:** MLP boleh saja mencapai akurasi tinggi di val — tapi data inipunya garis di POSISI ACAK. Lihat eksperimen Bagian 6: apa yang terjadi kalaugaris-geser muncul di data yang tidak pernah dilihat model?

## Bagian 2 — CNN Pertamamu

In [ ]:
from nn_mini.model import CNNMini

cnn = CNNMini(seed=0)
hist_cnn = train_model(cnn, X_train, y_train, X_val, y_val,
                       epochs=30, batch_size=32, lr=0.5, seed=0)
print(f'CNN mini: val acc = {hist_cnn["val_acc"][-1]:.4f}')

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_mlp['train_loss'], label='MLP train')
plt.plot(hist_mlp['val_loss'], '--', label='MLP val')
plt.plot(hist_cnn['train_loss'], label='CNN train')
plt.plot(hist_cnn['val_loss'], '--', label='CNN val')
plt.xlabel('epoch'); plt.ylabel('BCE'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist_mlp['val_acc'], label='MLP')
plt.plot(hist_cnn['val_acc'], label='CNN')
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.title('Akurasi'); plt.legend()
plt.tight_layout(); plt.show()


**Refleksi 2:** CNN mencapai akurasi tinggi dengan LEBIH SEDIKIT parameter(9 bobot filter + 17 dense vs ±1.300 MLP). Kenapa weight sharing begitu efektifuntuk pola yang posisinya tidak tetap?

## Bagian 3 — Membedah Model: Apa yang Dipelajari Filter?

In [ ]:
# Visualisasikan filter yang sudah DILATIH + responsnya terhadap dua kelas
K = cnn.conv.K
print('Filter 3x3 hasil training:\n', np.round(K, 2))

from nn_mini.conv import pad_zero, conv2d_forward, maxpool2x2_forward

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for k, kelas in enumerate([0, 1]):
    img = X_val[y_val == kelas][0]
    resp = conv2d_forward(pad_zero(img[None], 1), K)[0]
    axes[k, 0].imshow(img, cmap='gray'); axes[k, 0].set_title('input')
    axes[k, 1].imshow(resp, cmap='viridis'); axes[k, 1].set_title('setelah conv')
    axes[k, 2].imshow(np.maximum(resp, 0), cmap='viridis'); axes[k, 2].set_title('setelah ReLU')
    axes[k, 3].imshow(maxpool2x2_forward(np.maximum(resp, 0)[None])[0], cmap='viridis')
    axes[k, 3].set_title('setelah max-pool')
    for j in range(4):
        axes[k, j].axis('off')
plt.suptitle('Aliran informasi: conv → ReLU → pool')
plt.tight_layout(); plt.show()


**Refleksi 3:** bandingkan peta aktivasi kedua kelas. Di mana perbedaannya palingbesar? Nilai tunggal mana yang akhirnya 'membaca' dense layer? (hint: nilai MAKSIMUM)

## Bagian 4 — Eksperimen Ukuran Filter (satu variabel per percobaan)

In [ ]:
# TODO: jalankan CNNMini dengan kernel lain: 1x1, 2x2, 5x5 (pad disesuaikan
# supaya output tetap 4x4 setelah pool: pad = (k-1)//2).
# Catat val acc — apakah lebih besar selalu lebih baik?

# for kh in [1, 2, 3, 5]:
#     m = CNNMini(kh=kh, kw=kh, pad=(kh - 1) // 2, seed=0)
#     h = train_model(m, X_train, y_train, X_val, y_val,
#                     epochs=30, batch_size=32, lr=0.5, seed=0)
#     print(f'kernel {kh}x{kh}: val acc = {h["val_acc"][-1]:.4f}')


**Refleksi 4:** kernel 1x1 tidak bisa melihat tetangga — kenapa dia tetap bisabelajar (atau gagal)? Kapan filter besar justru merugikan? (ingat: berapa parameter,
berapa banyak piksel tepi yang hilang tanpa padding)

## Bagian 5 — Augmentasi: Flip, Noise, Shift

In [ ]:
from nn_mini.augment import augment_batch, random_shift

# TODO: bandingkan training TANPA vs DENGAN augmentasi tiap epoch.
# Pipeline: tiap epoch sebelum training, augment salinan batch data.
# Perhatikan: augmentasi HANYA di training — val/test tetap apa adanya!

# rng = np.random.default_rng(0)
# X_aug, y_aug = augment_batch(X_train, y_train, rng, p_noise=0.15)
# cnn_aug = CNNMini(seed=0)
# hist_aug = train_model(cnn_aug, X_aug, y_aug, X_val, y_val,
#                        epochs=30, batch_size=32, lr=0.5, seed=0)
# print(f'CNN + augmentasi: val acc = {hist_aug["val_acc"][-1]:.4f}')


**Refleksi 5:** augmentasi mana yang membantu di dataset ini, mana yang tidak?Ingat aturan Bab 5: augmentasi harus masuk akal untuk domain. Kenapa fliphorizontal aman untuk garis, tapi mengubah 'horizontal' jadi 'vertikal' justru
MERUSAK label? (jangan lakukan flip vertikal di dataset ini!)

## Bagian 6 — Ujian Sesungguhnya: Garis di Posisi Baru

In [ ]:
# TODO: uji ketahanan — geser SEMUA gambar val beberapa piksel, bandingkan MLP vs CNN.
# Model yang belajar 'bentuk' (CNN) harus jauh lebih tahan daripada yang
# mengahmali 'posisi' (MLP).

# from nn_mini.augment import random_shift
# rng = np.random.default_rng(7)
# for geser in [1, 2, 3]:
#     Xv = random_shift(X_val, rng, max_shift=geser)
#     acc_mlp = float(((mlp.predict_proba(Xv.reshape(len(Xv), -1)) >= 0.5) == y_val).mean())
#     acc_cnn = float(((cnn.predict_proba(Xv) >= 0.5) == y_val).mean())
#     print(f'geser ±{geser}px: MLP = {acc_mlp:.3f} | CNN = {acc_cnn:.3f}')


## Bagian 7 — Test Set (SEKALI, di akhir)

In [ ]:
# TODO: pilih model TERBAIKMU berdasarkan val, baru sentuh test set:
# p_test = cnn.predict_proba(X_test)
# acc_test = float(((p_test >= 0.5) == y_test).mean())
# print(f'TEST accuracy: {acc_test:.4f}')
#
# Lihat 5 prediksi pertama:
# for i in range(5):
#     pred = int(p_test[i] >= 0.5)
#     print(f'sampel {i}: prediksi={NILAI_KELAS[pred]}, benar={NILAI_KELAS[y_test[i]]}, '
#           f'p={p_test[i]:.2f}', '✓' if pred == y_test[i] else '✗')


---Selesai? Isi `RUBRIK.md` dengan jujur, baru bandingkan dengan `solusi.ipynb`dan `solusi/cnn_mini_ref.py`.